# Day 90: Double Machine Learning and Cross-Fitting

*Beginner to Ninja: Math for Causal Inference*

Orthogonal scores remove first-order sensitivity to nuisance errors, but a second problem remains: flexible learners fitted on the same observations they are evaluated on overfit, and their in-sample residuals carry the noise we are trying to measure. Cross-fitting (fit on some folds, residualise on the others, rotate) removes this own-observation bias without wasting data. Orthogonality plus cross-fitting is double machine learning (DML).

**Learning outcomes**

- Decompose the error of a plug-in estimator into a leading term, a nuisance-product term and an overfitting term.
- Explain why sample splitting kills the overfitting term and why cross-fitting restores full efficiency.
- Implement DML for the partially linear model with K-fold cross-fitting and compute its standard error.
- Aggregate estimates over repeated sample splits.

## Theory

### Three sources of error

For the partialling-out estimator with fitted nuisances $\hat\ell, \hat m$, write $v = D - m(X)$, $\varepsilon = Y - \ell(X) - \theta v$. Expanding $\sqrt n(\hat\theta - \theta)$ gives, up to a constant,

$$
\underbrace{\tfrac{1}{\sqrt n}\textstyle\sum v_i\varepsilon_i}_{\text{leading term}} \;+\; \underbrace{\sqrt n\, E_n\big[(m - \hat m)(\ell - \hat\ell)\big]}_{\text{product of errors}} \;+\; \underbrace{\tfrac{1}{\sqrt n}\textstyle\sum (\hat\ell - \ell)(X_i)\,v_i + \dots}_{\text{empirical-process (overfitting) terms}}.
$$

Orthogonality (Day 89) handles the product term when the nuisances converge faster than $n^{-1/4}$. The last term is the problem: if $\hat\ell$ was fitted using observation $i$, then $\hat\ell(X_i)$ partly memorises $Y_i$, and so is correlated with $v_i$.

### Sample splitting

If $\hat\ell$ is fitted on a separate sample, then conditional on that sample $(\hat\ell - \ell)(X_i)$ is a fixed function and $E[(\hat\ell - \ell)(X_i)v_i] = 0$. Its variance is at most $\|\hat\ell - \ell\|^2E[v^2]$, which goes to zero. No complexity restrictions (Donsker conditions) are needed, so any learner (forests, boosting, neural networks, lasso) is allowed.

### Cross-fitting

Splitting once wastes half the data. **Cross-fitting** splits the sample into $K$ folds $I_1, \dots, I_K$:

1. For each fold $k$, fit $\hat\ell^{(-k)}, \hat m^{(-k)}$ on the other folds.
2. Residualise observations in fold $k$: $\tilde Y_i = Y_i - \hat\ell^{(-k)}(X_i)$, $\tilde D_i = D_i - \hat m^{(-k)}(X_i)$.
3. **DML2** (pooled): $\hat\theta = \sum_i\tilde D_i\tilde Y_i/\sum_i\tilde D_i^2$ over all $i$. **DML1** averages the $K$ fold-specific estimates.

Every observation is used both for estimation and as an out-of-sample residual, so the estimator is as efficient as if the nuisances were known.

### Inference

With $\hat\psi_i = (\tilde Y_i - \hat\theta\tilde D_i)\tilde D_i$ and $\hat J = n^{-1}\sum\tilde D_i^2$,

$$
\widehat{\mathrm{SE}}(\hat\theta) = \sqrt{\frac{n^{-1}\sum\hat\psi_i^2}{\hat J^2\,n}},
$$

a heteroskedasticity-robust (sandwich) standard error. Because results depend on the random split, repeat it $S$ times and report the median estimate $\tilde\theta$ with variance $\mathrm{median}_s\{\hat\sigma_s^2 + (\hat\theta_s - \tilde\theta)^2\}$.

### Practical choices

$K = 5$ is standard; with clustered data, split by cluster; with time series, use blocked folds. Choose learners by out-of-sample prediction error for each nuisance separately.

## Part 1 · Warm-up

### Exercise 1 · Fold sizes

With $n = 2{,}000$ and $K = 5$, how many observations are used to train each nuisance fit, and how many are residualised with it?

<details><summary>Show solution</summary>

**Step 1.** Each fold has $400$ observations; each fit uses the other $1{,}600$.

**Answer:** train on 1,600, residualise 400; every observation is residualised exactly once.

</details>

*Your work:*

### Exercise 2 · The cost of a single split

A single 50/50 split uses half the sample for the final estimator. By how much does the variance rise compared with full-sample estimation?

<details><summary>Show solution</summary>

**Step 1.** Variance scales with $1/n$; using $n/2$ doubles it.

**Answer:** variance doubles (SE rises by about 41%); cross-fitting recovers this loss.

</details>

*Your work:*

### Exercise 3 · DML1 versus DML2

Fold estimates are $0.48$ and $0.54$ with $\sum\tilde D^2 = 300$ and $100$ respectively. Compute DML1 and DML2.

<details><summary>Show solution</summary>

**Step 1.** DML1: $(0.48 + 0.54)/2 = 0.51$.

**Step 2.** DML2 weights each fold by its $\sum\tilde D^2$: $(300\times0.48 + 100\times0.54)/400 = 0.495$.

**Answer:** 0.51 versus 0.495; DML2 is more stable when folds are small, which is why it is the default.

</details>

*Your work:*

### Exercise 4 · Why a fixed function helps

Given the training folds, $\hat\ell^{(-k)} - \ell = h$ is a fixed function. What is $E[h(X_i)v_i]$ for $i$ in fold $k$?

<details><summary>Show solution</summary>

**Step 1.** $E[h(X_i)v_i] = E[h(X_i)E[v_i \mid X_i]] = 0$, because $h$ does not depend on observation $i$.

</details>

*Your work:*

### Exercise 5 · Clusters

Patients are nested in 60 hospitals and outcomes are correlated within hospital. How should folds be formed?

<details><summary>Show solution</summary>

**Step 1.** Assign whole hospitals to folds, so no hospital's patients are both training and evaluation data.

**Step 2.** Use cluster-robust standard errors in the final step.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Error decomposition

Let $\tilde Y = Y - \hat\ell(X)$, $\tilde D = D - \hat m(X)$, $\Delta_\ell = \ell - \hat\ell$, $\Delta_m = m - \hat m$. Show $E_n[\tilde D(\tilde Y - \theta\tilde D)] = E_n[v\varepsilon] + E_n[\Delta_m\varepsilon] + E_n[v(\Delta_\ell - \theta\Delta_m)] + E_n[\Delta_m(\Delta_\ell - \theta\Delta_m)]$.

<details><summary>Show solution</summary>

**Step 1.** $\tilde D = v + \Delta_m$ and $\tilde Y - \theta\tilde D = (\ell + \theta v + \varepsilon - \hat\ell) - \theta(v + \Delta_m) = \varepsilon + \Delta_\ell - \theta\Delta_m$.

**Step 2.** Multiply out $(v + \Delta_m)(\varepsilon + \Delta_\ell - \theta\Delta_m)$ and average. $\blacksquare$

The first term is the leading term; the middle two are overfitting terms; the last is the product of errors.

</details>

*Your work:*

### Exercise 7 · Cross-fitting kills the overfitting terms

For $i$ in fold $k$, with $\Delta_\ell$ fitted on the other folds, show $E_{I_k}[v\Delta_\ell]$ has conditional mean zero and conditional variance $\le E[v^2 \mid X]_{\max}\|\Delta_\ell\|^2/n_k$.

<details><summary>Show solution</summary>

**Step 1.** Conditional on the training folds, the terms $v_i\Delta_\ell(X_i)$ are i.i.d. with mean $E[\Delta_\ell(X)E[v \mid X]] = 0$.

**Step 2.** Variance of the mean $= E[v^2\Delta_\ell^2]/n_k \le \sup_x E[v^2 \mid X = x]\cdot\|\Delta_\ell\|_2^2/n_k$.

**Step 3.** So $\sqrt n E_{I_k}[v\Delta_\ell] = O_p(\|\Delta_\ell\|) \to 0$ whenever the learner is consistent. $\blacksquare$ No restriction on the learner's complexity is needed.

</details>

*Your work:*

### Exercise 8 · Overfitting attenuates the estimate

Suppose in-sample $\hat\ell(X_i)$ absorbs a fraction $a$ of the noise: $\tilde Y_i = (1 - a)(\theta v_i + \varepsilon_i)$, while $\hat m$ is smooth so $\tilde D_i = v_i$. Find the probability limit of $\hat\theta$.

<details><summary>Show solution</summary>

**Step 1.** $\hat\theta = \sum\tilde D\tilde Y/\sum\tilde D^2 = (1 - a)\big(\theta\sum v^2 + \sum v\varepsilon\big)/\sum v^2$.

**Step 2.** $\sum v\varepsilon/\sum v^2 \to 0$.

**Answer:** $\hat\theta \to (1 - a)\theta$: attenuated towards zero by the share of noise the outcome learner memorised.

</details>

*Your work:*

### Exercise 9 · DML2 as a weighted average

Show DML2 equals $\sum_k w_k\hat\theta_k$ with $w_k = \sum_{i \in I_k}\tilde D_i^2/\sum_i\tilde D_i^2$.

<details><summary>Show solution</summary>

**Step 1.** $\hat\theta_k = \sum_{I_k}\tilde D\tilde Y/\sum_{I_k}\tilde D^2$, so $\sum_{I_k}\tilde D\tilde Y = \hat\theta_k\sum_{I_k}\tilde D^2$.

**Step 2.** Sum over $k$ and divide by $\sum_i\tilde D_i^2$. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · Cross-fitting recovers full efficiency

Each fold estimate is asymptotically $\hat\theta_k - \theta \approx J^{-1}E_{I_k}[v\varepsilon]$, with variance $K\sigma^2/n$, where $\sigma^2 = E[v^2\varepsilon^2]/J^2$. Show the average has variance $\sigma^2/n$.

<details><summary>Show solution</summary>

**Step 1.** The leading terms involve disjoint observations, so they are independent.

**Step 2.** $\mathrm{Var}\big(K^{-1}\sum_k\hat\theta_k\big) = K^{-2}\cdot K\cdot K\sigma^2/n = \sigma^2/n$.

**Answer:** the same as the infeasible estimator using the true nuisances on all $n$ observations. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · Community health worker visits and child health

A district varies the number of community health worker visits $D$ across 500 households; visits depend nonlinearly on 10 household characteristics, which also affect a child health index $Y$. True effect: 0.5 per visit. The outcome model is a deep random forest (leaves of size 1); the treatment model a smoother forest. Compare in-sample residualisation with 5-fold cross-fitting.

<details><summary>Show solution</summary>

**Step 1.** In-sample: mean estimate about 0.22, coverage of the nominal 95% interval 0%. The deep forest memorises much of $Y_i$, including the part due to $v_i$, so the effect is attenuated (Exercise 8).

**Step 2.** Cross-fitted: mean about 0.48, with coverage close to the nominal level (about 88% in 40 replications).

**Answer:** the same learners give a badly biased estimate without cross-fitting and a nearly unbiased one with it.

</details>

*Your work:*

### Exercise 12 · Repeated splits

Five random splits of a study of a sugar tax on soda purchases give estimates $-0.48, -0.52, -0.50, -0.47, -0.55$ litres per week, each with SE 0.05. Report the aggregated estimate and SE.

<details><summary>Show solution</summary>

**Step 1.** Median estimate: $-0.50$.

**Step 2.** $\hat\sigma_s^2 + (\hat\theta_s - \tilde\theta)^2$: $0.0029, 0.0029, 0.0025, 0.0034, 0.0050$.

**Step 3.** Median: $0.0029$, so SE $= \sqrt{0.0029} = 0.054$.

**Answer:** $-0.50$ litres/week (SE 0.054); the extra spread across splits widens the SE slightly.

</details>

*Your work:*

### Exercise 13 · Choosing K

With $n = 300$, a colleague proposes $K = 2$. Why might $K = 5$ or $10$ be better, and when is $K = 2$ fine?

<details><summary>Show solution</summary>

**Step 1.** With $K = 2$, nuisances are trained on only 150 observations, so they are less accurate, enlarging the product-of-errors term.

**Step 2.** Larger $K$ trains on $(K - 1)n/K$ observations; the cost is computation.

**Answer:** in small samples use $K = 5$–$10$; with very large $n$, $K = 2$ loses little.

</details>

*Your work:*

### Exercise 14 · Minimum wages across counties

A study of minimum wages and teen employment uses 3,000 county-quarters from 300 counties with lasso nuisances. List three implementation decisions that matter for valid inference.

<details><summary>Show solution</summary>

**Step 1.** Folds by county (clusters), not by observation.

**Step 2.** County-clustered standard errors in the final stage.

**Step 3.** Repeated splits with median aggregation, and reporting of nuisance prediction accuracy.

</details>

*Your work:*

### Exercise 15 · Reporting DML results

Draft a two-sentence methods description for a DML analysis of a diabetes education programme's effect on HbA1c.

<details><summary>Show solution</summary>

**Step 1.** "We estimate the effect using double machine learning (Chernozhukov et al., 2018) for the partially linear model, with random-forest and lasso nuisance models chosen by cross-validated prediction error and 5-fold cross-fitting."

**Step 2.** "Estimates are medians over 20 random splits; standard errors are heteroskedasticity-robust and account for variation across splits."

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

rng = np.random.default_rng(90)
def gen(n, p=10):
    X = rng.normal(size=(n, p))
    m = np.tanh(X[:, 0]) + X[:, 1] ** 2 / 4
    g = np.sin(2 * X[:, 0]) + X[:, 2] ** 2 / 2 + X[:, 1]
    D = m + rng.normal(size=n); Y = .5 * D + g + rng.normal(size=n)
    return X, D, Y
rf = lambda leaf: RandomForestRegressor(n_estimators=40, min_samples_leaf=leaf, max_features=.5, n_jobs=-1, random_state=0)

def dml(X, D, Y, cross):
    if cross:
        rY, rD = np.empty_like(Y), np.empty_like(D)
        for tr, te in KFold(5, shuffle=True, random_state=1).split(X):
            rY[te] = Y[te] - rf(1).fit(X[tr], Y[tr]).predict(X[te])      # deep forest for the outcome
            rD[te] = D[te] - rf(10).fit(X[tr], D[tr]).predict(X[te])     # smoother forest for treatment
    else:
        rY = Y - rf(1).fit(X, Y).predict(X); rD = D - rf(10).fit(X, D).predict(X)
    th = rD @ rY / (rD @ rD); psi = (rY - th * rD) * rD
    return th, np.sqrt(np.mean(psi ** 2) / np.mean(rD ** 2) ** 2 / len(Y))

for cross in (False, True):
    r = np.array([dml(*gen(500), cross) for _ in range(40)])
    cov = np.mean(np.abs(r[:, 0] - .5) < 1.96 * r[:, 1])
    print(f"{'cross-fit' if cross else 'in-sample'}: mean {r[:, 0].mean():.3f}  SD {r[:, 0].std():.3f}  mean SE {r[:, 1].mean():.3f}  coverage {cov:.2f}")

# Exercise 12: repeated-split aggregation
th = np.array([-.48, -.52, -.50, -.47, -.55]); med = np.median(th)
print("median", med, " SE", round(float(np.sqrt(np.median(.05 ** 2 + (th - med) ** 2))), 4))

## Key takeaways

- Learners fitted on the same observations they residualise overfit, and the overfitting bias does not vanish even with an orthogonal score.
- Cross-fitting (train on other folds, residualise this fold, rotate) removes this bias and keeps full efficiency, with no restrictions on the learner.
- Report robust standard errors, split by cluster where relevant, and aggregate over repeated splits.